# 04 — Travel Planner: Multi-Agent Collaboration Pipeline

### Core concepts covered
| Concept | Description |
|---------|-------------|
| Multi-agent pipeline | Multiple specialised agents, each with one role |
| State as shared memory | Each agent reads prior agents' outputs as context |
| Final compiler node | A node that assembles all outputs into one document |

### Agents in the pipeline
```
START → destination_researcher → budget_planner → itinerary_writer → compile_plan → END
```

> **Before running:** set `GROQ_API_KEY` in a `.env` file in this folder.

## 1. Install Dependencies

In [ ]:
# !pip install langgraph langchain langchain-groq python-dotenv

## 2. Imports & Setup

In [ ]:
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model("groq:llama3-8b-8192")

## 3. Define the State

In [ ]:
class TravelState(TypedDict):
    destination:      str   # e.g. "Tokyo, Japan"
    duration_days:    int   # number of days
    budget_usd:       int   # total budget
    travel_style:     str   # budget | comfort | luxury
    destination_info: str   # filled by agent 1
    budget_breakdown: str   # filled by agent 2
    itinerary:        str   # filled by agent 3
    final_plan:       str   # compiled by the last node

## 4. Define the Agents

Each agent is a node that does one job and passes a richer state forward.

In [ ]:
def destination_researcher(state: TravelState) -> dict:
    """Agent 1 — research the destination and gather practical travel info."""
    print(f"\n  🔍 Researching {state['destination']}...")
    prompt = (
        f"You are a seasoned travel expert.\n"
        f"Provide practical travel information about {state['destination']} "
        f"for a {state['duration_days']}-day trip.\n\n"
        "Cover:\n"
        "1. Best time to visit & weather\n"
        "2. Top 5 must-see attractions (one line each)\n"
        "3. Must-try local food or drink\n"
        "4. Key cultural tips for visitors\n"
        "5. How to get around locally\n\n"
        "Be concise and practical."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  ✅ Destination research done.")
    return {"destination_info": response.content.strip()}

In [ ]:
def budget_planner(state: TravelState) -> dict:
    """Agent 2 — create a realistic budget breakdown."""
    print(f"\n  💰 Planning budget (${state['budget_usd']} / {state['travel_style']} style)...")
    prompt = (
        f"You are a travel budget specialist.\n"
        f"Create a detailed budget breakdown for this trip:\n\n"
        f"  Destination   : {state['destination']}\n"
        f"  Duration      : {state['duration_days']} days\n"
        f"  Total budget  : ${state['budget_usd']} USD\n"
        f"  Travel style  : {state['travel_style']}\n\n"
        "Break it down by: Accommodation, Meals, Activities, Local transport, Buffer.\n"
        "Show per-day averages, category totals, and grand total.\n"
        "Flag clearly if the budget is tight or generous."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  ✅ Budget plan done.")
    return {"budget_breakdown": response.content.strip()}

In [ ]:
def itinerary_writer(state: TravelState) -> dict:
    """Agent 3 — write a day-by-day itinerary using outputs from agents 1 & 2."""
    print(f"\n  📅 Writing {state['duration_days']}-day itinerary...")
    prompt = (
        f"You are a professional tour planner.\n"
        f"Write a day-by-day itinerary for {state['destination']} ({state['duration_days']} days, "
        f"{state['travel_style']} style).\n\n"
        f"Key highlights:\n{state['destination_info'][:600]}\n\n"
        f"Budget context:\n{state['budget_breakdown'][:400]}\n\n"
        "Format each day as:\n"
        "  Day N — <theme>\n"
        "  Morning   : ...\n"
        "  Afternoon : ...\n"
        "  Evening   : ...\n"
        "  Tip       : one practical tip"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  ✅ Itinerary written.")
    return {"itinerary": response.content.strip()}

In [ ]:
def compile_plan(state: TravelState) -> dict:
    """Final node — assemble all agent outputs into one travel document."""
    plan = (
        f"\n{'═'*58}\n"
        f"           ✈️   YOUR COMPLETE TRAVEL PLAN   ✈️\n"
        f"{'═'*58}\n"
        f"  Destination  : {state['destination']}\n"
        f"  Duration     : {state['duration_days']} days\n"
        f"  Budget       : ${state['budget_usd']} USD  ({state['travel_style']} style)\n"
        f"{'═'*58}\n\n"
        f"📍  DESTINATION OVERVIEW\n{'─'*45}\n"
        f"{state['destination_info']}\n\n"
        f"💰  BUDGET BREAKDOWN\n{'─'*45}\n"
        f"{state['budget_breakdown']}\n\n"
        f"🗓️   DAY-BY-DAY ITINERARY\n{'─'*45}\n"
        f"{state['itinerary']}\n\n"
        f"{'═'*58}\n"
        f"  Have a wonderful trip! 🌍\n"
        f"{'═'*58}"
    )
    print(plan)
    return {"final_plan": plan}

## 5. Build & Compile the Graph

In [ ]:
builder = StateGraph(TravelState)

builder.add_node("destination_researcher", destination_researcher)
builder.add_node("budget_planner",         budget_planner)
builder.add_node("itinerary_writer",       itinerary_writer)
builder.add_node("compile_plan",           compile_plan)

builder.add_edge(START,                    "destination_researcher")
builder.add_edge("destination_researcher", "budget_planner")
builder.add_edge("budget_planner",         "itinerary_writer")
builder.add_edge("itinerary_writer",       "compile_plan")
builder.add_edge("compile_plan",           END)

graph = builder.compile()
print("Graph compiled!")

## 6. Visualise the Graph *(optional)*

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualisation skipped: {e}")

## 7. Run the Travel Planner

Fill in your destination, trip length, budget, and style below.

In [ ]:
destination = input("Destination (e.g., Tokyo, Paris, Bali): ").strip() or "Tokyo, Japan"

days_input  = input("Duration in days (e.g., 5): ").strip()
duration    = int(days_input) if days_input.isdigit() else 5

budget_input = input("Total budget in USD (e.g., 2000): ").strip()
budget       = int(budget_input) if budget_input.isdigit() else 2000

style = input("Travel style — budget / comfort / luxury (default: comfort): ").strip().lower()
if style not in ("budget", "comfort", "luxury"):
    style = "comfort"

print(f"\n🚀 Building your travel plan for {destination}...\n")

In [ ]:
graph.invoke({
    "destination":      destination,
    "duration_days":    duration,
    "budget_usd":       budget,
    "travel_style":     style,
    "destination_info": "",
    "budget_breakdown": "",
    "itinerary":        "",
    "final_plan":       "",
})